# Freyra — Influencer Edition (T4 Colab)
Photorealistic female influencer images on a free T4 GPU.  
Run **Cell 2** (setup), then **Cell 3** (launch). That's it.


In [ ]:
# ── Cell 2 · Setup ───────────────────────────────────────────────────────
%cd /content
!git clone https://github.com/Ashutosh94g/Freyra.git 2>/dev/null || git -C Freyra pull --ff-only
%cd /content/Freyra

# Remove cupy if present — it forces numpy 2.x which breaks the pipeline
!pip uninstall -yq cupy-cuda12x cupy-cuda11x cupy 2>/dev/null; true

# PyTorch 2.4.1 + CUDA 12.1 (verified on T4)
%pip install -q torch==2.4.1 torchvision==0.19.1 --extra-index-url https://download.pytorch.org/whl/cu121

# All project requirements (gradio 3.50.2, transformers>=4.45, numpy<2.0, etc.)
%pip install -q -r requirements_versions.txt

print("\n✅ Setup complete — run Cell 3 to launch.")


In [ ]:
# ── Cell 4 · Verify GPU & VRAM before launch ─────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb  = props.total_memory / 1024**3
    free_gb   = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f'GPU  : {props.name}')
    print(f'VRAM : {total_gb:.1f} GB total  |  {free_gb:.1f} GB free')
    if total_gb < 12:
        print('⚠️  Less than 12 GB VRAM — you may need to lower resolution in the UI.')
    else:
        print('✅ VRAM looks fine for influencer preset.')
else:
    print('❌ No CUDA GPU found. Are you on a GPU runtime? (Runtime → Change runtime type → T4)')

In [ ]:
# ── Cell 5 · Launch Freyra (influencer preset) ────────────────────────────
#
#  --share               public Gradio tunnel URL
#  --always-gpu          keep all models on GPU (avoids slow CPU<->GPU swap)
#  --unet-in-fp8-e4m3fn  compress UNet VRAM 6.5 GB → 3.3 GB
#  --attention-pytorch   use PyTorch SDPA (no xformers needed on T4)
#  --preset influencer   loads presets/influencer.json
#  --disable-image-log   saves a little RAM on Colab's 12.7 GB system limit
#
import os
os.chdir('/content/Freyra')
os.environ['GRADIO_SERVER_PORT'] = '7865'

os.system(
    'python launch.py '
    '--always-gpu '
    '--always-high-vram '
    '--attention-pytorch '
    '--preset influencer '
    '--disable-image-log'

))

## Troubleshooting

| Symptom | Fix |
|---|---|
| `RuntimeError: CUDA out of memory` | Lower resolution to `832×1104` in the UI, or enable only 1 LoRA |
| `ModuleNotFoundError: numpy` | Re-run Cell 2 — runtime may have reset |
| Gradio URL not appearing | Wait 60–90 seconds; large model download may still be in progress |
| `pygit2` version error | Cell 2 re-installs it; restart runtime after Cell 2 if needed |
| Black/broken images | Check the LoRA strength — try reducing Film Photography LoRA to 0.15 |

### Resolution guide for T4

| Aspect ratio | Safe resolution | Notes |
|---|---|---|
| Portrait 3:4 | `896×1152` | Default influencer preset |
| Square 1:1 | `1024×1024` | Fine with fp8 UNet |
| Landscape 4:3 | `1152×896` | Fine |
| Tall portrait 2:3 | `832×1248` | Reduce LoRA if OOM |